In [8]:
import threading

def make_coffee():
    name = threading.current_thread().name
    print(name)
    print(threading.active_count()) # 7

# Creating thread
coffee_thread = threading.Thread(target=make_coffee, name="coffee_thread")

# starting a thread
coffee_thread.start()

# joining the thread back to main thread at an appropriate place
coffee_thread.join()


coffee_thread
7


In [10]:
import random
import time

# synchronous code

def send_email(name):
    print(f"sending email to: {name}")
    time.sleep(random.randint(1,3))
    print(f"email to {name} sent successfully")

def send_msg(name):
    print(f"sending msg to: {name}")
    time.sleep(random.randint(1,3))
    print(f"msg to {name} sent successfully")

friends = ['ram', 'shyam', 'babu']

start = time.time()

for friend in friends:
    send_email(friend)
    send_msg(friend)

time_taken = time.time() - start

print(f"Total time taken: {time_taken:.2f} seconds")


sending email to: ram
email to ram sent successfully
sending msg to: ram
msg to ram sent successfully
sending email to: shyam
email to shyam sent successfully
sending msg to: shyam
msg to shyam sent successfully
sending email to: babu
email to babu sent successfully
sending msg to: babu
msg to babu sent successfully
Total time taken: 14.01 seconds


In [15]:
# multithreading code

def send_email(name):
    print(f"sending email to: {name}")
    time.sleep(random.randint(1,3))
    print(f"email to {name} sent successfully")

def send_msg(name):
    print(f"sending msg to: {name}")
    time.sleep(random.randint(1,3))
    print(f"msg to {name} sent successfully")

friends = ['ram', 'shyam', 'babu']

start = time.time()

# create
threads = []
for friend in friends:
    threads.append(threading.Thread(target=send_email, args=(friend, ), name=f"email_{friend}"))
    threads.append(threading.Thread(target=send_msg, args=(friend, ), name=f"msg_{friend}"))

# start
for thread in threads:
    thread.start()

# join
for thread in threads:
    thread.join()

time_taken = time.time() - start

print(f"Total time taken: {time_taken:.2f} seconds")


sending email to: ram
sending msg to: ram
sending email to: shyam
sending msg to: shyam
sending email to: babu
sending msg to: babu
email to shyam sent successfullyemail to babu sent successfully

msg to ram sent successfully
email to ram sent successfully
msg to babu sent successfully
msg to shyam sent successfully
Total time taken: 3.01 seconds


In [ ]:
threading.enumerate() # show all the thread running the the system known by python


[<_MainThread(MainThread, started 25444)>,
 <Thread(IOPub, started daemon 8908)>,
 <Heartbeat(Heartbeat, started daemon 33448)>,
 <ControlThread(Control, started daemon 11184)>,
 <HistorySavingThread(IPythonHistorySavingThread, started 19032)>,
 <ParentPollerWindows(Thread-1, started daemon 25940)>]

# Race Conditon

### Shared memory in threads

In [6]:
import threading

shared_list = []

def append_items(tag):
    for i in range(3):
        shared_list.append(f"{tag}{i}")

t1 = threading.Thread(target=append_items, args=("A",))
t2 = threading.Thread(target=append_items, args=("B", ))

t1.start()
t2.start()

t1.join()
t2.join()

print(shared_list)

['A0', 'A1', 'A2', 'B0', 'B1', 'B2']


### What is a race condition?

The central problem is simple:

Multiple threads can access the same piece of the shared state, and the final resilt can depend on the timing and interleaving of their execution.

That situation is called a race condition

In [ ]:
### Simulate a race condition

import threading
import time
import random

counter = 0
iters = 20

def increment_many():
    global counter
    for _ in range(iters):
        # critical block: read, update, write
        current = counter
        time.sleep(random.uniform(0.00001, 0.00005))
        counter = current + 1

num_threads = 4
threads = [threading.Thread(target=increment_many, name="increment_thread") for _ in range(num_threads)]

print(f"{len(threads) = }")

for t in threads:
    t.start()

for t in threads:
    t.join()

print(f"expected: {num_threads * iters}")
print(f"actual: {counter}")

len(threads) = 4
expected: 80
actual: 20


In [15]:
import threading, time

balance = 100

def withdraw(amount):
    global balance
    if balance >= amount: # CHECK
        time.sleep(0.001) # any pause at all: I/O, logging, GC, scheduler
        balance -= amount
        print(f"withdraw {amount}, balance now {balance}")
    else:
        print(f"declined {amount}")

t1 = threading.Thread(target=withdraw, args=(100,))
t2 = threading.Thread(target=withdraw, args=(100,))

t1.start()
t2.start()

t1.join()
t2.join()

print(f"final balance: {balance}")

withdraw 100, balance now 0
withdraw 100, balance now -100
final balance: -100


## What is a lock?

A lock is a synchronization mechanism that allows us to say:
"Only one thread at a time may enter this protected section."

In [17]:
import threading, time

balance = 100

lock = threading.Lock()

def withdraw(amount):
    global balance
    with lock:
        if balance >= amount: # CHECK
            time.sleep(0.001) # any pause at all: I/O, logging, GC, scheduler
            balance -= amount
            print(f"withdraw {amount}, balance now {balance}")
        else:
            print(f"declined {amount}")

t1 = threading.Thread(target=withdraw, args=(100,))
t2 = threading.Thread(target=withdraw, args=(100,))

t1.start()
t2.start()

t1.join()
t2.join()

print(f"final balance: {balance}")

withdraw 100, balance now 0
declined 100
final balance: 0


In [ ]:
lock = threading.Lock()
# with lock: default choice

# if need more control
lock.acquire(timeout=1, blocking=False)
lock.release()